# 04 — Spatial Units

Loads or generates spatial analysis units:
1. **Districts** — official Gdynia administrative districts
2. **Neighborhoods** — sub-district housing estates
3. **Grid** — regular 250 m cells as fallback

Outputs: `data/processed/spatial_units.gpkg`

In [ ]:
from __future__ import annotations

import pathlib
import geopandas as gpd
import matplotlib.pyplot as plt
from gdynia_thermal_audit.settings import Settings
from gdynia_thermal_audit.logging_utils import setup_logging
from gdynia_thermal_audit.spatial_units.districts import load_districts, get_gdynia_districts_placeholder
from gdynia_thermal_audit.spatial_units.grid import generate_grid, clip_grid_to_boundary
from gdynia_thermal_audit.constants import GDYNIA_BBOX_WGS84, EPSG_POLAND
from gdynia_thermal_audit.utils.io import ensure_dir

setup_logging()
settings = Settings()
PROCESSED_DIR = pathlib.Path(settings.data_dir) / 'processed'
ensure_dir(PROCESSED_DIR)

In [ ]:
# Try loading districts from config; fall back to demo or placeholder
districts_path = 'data/demo/demo_districts.geojson'
try:
    gdf_districts = load_districts(districts_path)
    print(f'Loaded {len(gdf_districts)} districts')
except Exception as e:
    print(f'Could not load districts ({e}); using placeholder')
    gdf_districts = get_gdynia_districts_placeholder()

print(gdf_districts.crs)
gdf_districts.head()

In [ ]:
# Generate 250m grid covering Gdynia bbox in EPSG:2180
import pyproj
from shapely.geometry import box
from pyproj import Transformer

# Transform WGS84 bbox to EPSG:2180
transformer = Transformer.from_crs('EPSG:4326', f'EPSG:{EPSG_POLAND}', always_xy=True)
xmin, ymin = transformer.transform(GDYNIA_BBOX_WGS84[0], GDYNIA_BBOX_WGS84[1])
xmax, ymax = transformer.transform(GDYNIA_BBOX_WGS84[2], GDYNIA_BBOX_WGS84[3])
bbox_2180 = (xmin, ymin, xmax, ymax)

gdf_grid = generate_grid(bbox=bbox_2180, cell_size_m=250, epsg=EPSG_POLAND)
print(f'Generated {len(gdf_grid)} grid cells')
gdf_grid.head()

In [ ]:
# Save to GeoPackage
gdf_districts_2180 = gdf_districts.to_crs(epsg=EPSG_POLAND)
gdf_districts_2180.to_file(PROCESSED_DIR / 'spatial_units.gpkg', layer='districts', driver='GPKG')
gdf_grid.to_file(PROCESSED_DIR / 'spatial_units.gpkg', layer='grid_250m', driver='GPKG')
print('Spatial units saved to data/processed/spatial_units.gpkg')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
gdf_districts_2180.plot(ax=axes[0], edgecolor='black', facecolor='lightblue', alpha=0.5)
axes[0].set_title('Gdynia Districts')
gdf_grid.plot(ax=axes[1], edgecolor='grey', facecolor='none', linewidth=0.4)
gdf_districts_2180.plot(ax=axes[1], edgecolor='black', facecolor='none', linewidth=1.5)
axes[1].set_title('250 m Grid + Districts')
plt.tight_layout()
plt.savefig('outputs/figures/spatial_units.png', dpi=120, bbox_inches='tight')
plt.show()